In [2]:
# Uninstall the currently installed versions of these packages
!pip uninstall -y transformers sentence-transformers accelerate tokenizers

# Install specific compatible versions of the required packages
!pip install \
transformers==4.45.2 \
sentence-transformers==3.1.1 \
accelerate==0.34.2 \
tokenizers==0.20.1 \
datasets==2.21.0 \
numpy==1.26.4

In [2]:
# Import the function used to download and load datasets
from datasets import load_dataset

# Load the Multi-Genre Natural Language Inference (MNLI) dataset from the GLUE benchmark
# Labels:
# 0 = Entailment
# 1 = Neutral
# 2 = Contradiction
train_dataset = load_dataset(
    "glue",
    "mnli",
    split="train"
).select(range(50_000))

# Remove the "idx" column because it is only an identifier and is not useful for training
train_dataset = train_dataset.remove_columns("idx")

In [ ]:
# Import the SentenceTransformer class for creating sentence embeddings
from sentence_transformers import SentenceTransformer

# Load the pre-trained BERT base uncased model as the embedding model
embedding_model = SentenceTransformer('bert-base-uncased')

In [ ]:
# Import the loss functions available in Sentence Transformers
from sentence_transformers import losses

# Create the Softmax Loss used to fine-tune the embedding model
train_loss = losses.SoftmaxLoss(

    # The SentenceTransformer model to be trained
    model=embedding_model,

    # Size of the sentence embedding produced by the model
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),

    # Number of output classes (MNLI has 3 labels)
    num_labels=3
)

In [ ]:
# Import the evaluator used to measure sentence embedding quality
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the validation split of the STS-B (Semantic Textual Similarity Benchmark) dataset
val_sts = load_dataset("glue", "stsb", split="validation")

# Create an evaluator to compare sentence embeddings using cosine similarity
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence of each sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence of each sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize similarity scores from 0–5  to 0–1
    # The STS-B labels look like this(0 1.5 2.7 4.3 5 )
    # SentenceTransformers expects 0-1 so they are normalized by dividing with 5
    scores=[score / 5 for score in val_sts["label"]],

    # Use cosine similarity to compare embeddings
    main_similarity="cosine",
)

In [ ]:
# Import the training arguments class for Sentence Transformers
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define all the training configuration (hyperparameters)
args = SentenceTransformerTrainingArguments(

    # Folder where the trained model and checkpoints will be saved
    output_dir="base_embedding_model",

    # Train the model for 1 complete pass through the dataset
    num_train_epochs=1,

    # Number of training samples processed on each device in one step
    per_device_train_batch_size=32,

    # Number of validation samples processed on each device in one step
    per_device_eval_batch_size=32,

    # Gradually increase the learning rate during the first 100 training steps
    warmup_steps=100,

    # Use 16-bit floating point precision to reduce GPU memory usage and speed up training
    fp16=True,

    # Evaluate the model after every 100 training steps
    eval_steps=100,

    # Print training logs after every 100 training steps
    logging_steps=100,
)

In [ ]:
# Import the Trainer class from the Transformers library
from transformers import Trainer

# Import Python's built-in inspect module
# It allow us to inspect functions ,methods ,parameters
import inspect

# Print the function signature (parameters) of the training_step method
#The signature() function shows what parameters a function accepts
print(inspect.signature(Trainer.training_step))

In [ ]:
# Import Python's inspect module
import inspect

# Import the SentenceTransformerTrainer class
from sentence_transformers import SentenceTransformerTrainer

# Display the parameter list (function signature) of the compute_loss() method
print(inspect.signature(SentenceTransformerTrainer.compute_loss))

In [ ]:
# Import the SentenceTransformerTrainer class
from sentence_transformers.trainer import SentenceTransformerTrainer

# Create the trainer by providing the model, training settings,
# training dataset, loss function, and evaluation method
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)

# Start fine-tuning the sentence embedding model
trainer.train()

In [ ]:
# Evaluate the fine-tuned embedding model on the STS-B validation dataset
evaluator(embedding_model)

In [ ]:
# Install version 1.1.2 of the Massive Text Embedding Benchmark (MTEB) library
# It is a benchmarking library used to evaluate sentence embedding models on many NLP tasks.
!pip install mteb==1.1.2

In [ ]:
# Import the MTEB benchmark class
from mteb import MTEB

# Select the Banking77 text classification benchmark
evaluation = MTEB(
    tasks=["Banking77Classification"]
)

# Evaluate the sentence embedding model on the selected benchmark
# We are  trained embedding model is tested on Banking77 Dataset .
results = evaluation.run(embedding_model)

In [ ]:
# Import Dataset class and function to load datasets
from datasets import Dataset, load_dataset

# Load the first 50,000 examples from the MNLI training dataset
train_dataset = load_dataset(
    "glue", "mnli", split="train"
).select(range(50_000))

# Remove the unnecessary 'idx' column
train_dataset = train_dataset.remove_columns("idx")

# Create a mapping for binary classification
# Original labels:
# 0 = Entailment
# 1 = Neutral
# 2 = Contradiction
#
# New labels:
# Entailment = 1
# Neutral + Contradiction = 0
mapping = {2: 0, 1: 0, 0: 1}

# Create a new dataset with only the required columns
train_dataset = Dataset.from_dict({

    # Rename premise to sentence1
    # Sentence Transformers expect sentence1, sentence2 instead of premise and hypothesis
    "sentence1": train_dataset["premise"],

    # Rename hypothesis to sentence2
    "sentence2": train_dataset["hypothesis"],

    # Convert the original labels into binary labels
    # We use float because some loss functions used in Sentence Transformers expect labels as floating-point values instead of integers.
    "label": [
        float(mapping[label])
        for label in train_dataset["label"]
    ]
})

In [ ]:
# Import the evaluator used to measure sentence embedding quality
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the STS-B (Semantic Textual Similarity Benchmark) validation dataset
val_sts = load_dataset("glue", "stsb", split="validation")

# Create an evaluator that compares predicted sentence similarity
# with the actual similarity scores from the dataset
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence in each sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence in each sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize similarity scores from the range 0–5 to 0–1
    scores=[
        score / 5
        for score in val_sts["label"]
    ],

    # Use cosine similarity to compare sentence embeddings
    main_similarity="cosine"
)

In [ ]:
# Import the loss functions and SentenceTransformer model
from sentence_transformers import losses, SentenceTransformer

# Import the trainer used for training SentenceTransformer models
from sentence_transformers.trainer import SentenceTransformerTrainer

# Import training configuration arguments
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Load the pretrained BERT model as a SentenceTransformer
embedding_model = SentenceTransformer("bert-base-uncased")

# Use Cosine Similarity Loss for training
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define training configuration
args = SentenceTransformerTrainingArguments(
    output_dir="cosineloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Create the trainer
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)

# Start training
trainer.train()

# Evaluate the trained model on the STS-B validation dataset
evaluator(embedding_model)

In [ ]:
# Import Python's random module
import random

# Import tqdm to display a progress bar
from tqdm import tqdm

# Import Dataset class and function to load datasets
from datasets import Dataset, load_dataset

# Load the first 50,000 examples from the MNLI training dataset
mnli = load_dataset(
    "glue",
    "mnli",
    split="train"
).select(range(50_000))

# Remove the unnecessary index column
mnli = mnli.remove_columns("idx")

# Keep only entailment examples (label = 0)
mnli = mnli.filter(
    lambda x: True if x["label"] == 0 else False
)

# Create empty lists for triplet training data
train_dataset = {
    "anchor": [],
    "positive": [],
    "negative": []
}

# Copy all hypotheses to create soft negatives
soft_negatives = mnli["hypothesis"]

# Shuffle them randomly
random.shuffle(soft_negatives)

# Build anchor-positive-negative triplets
for row, soft_negative in tqdm(zip(mnli, soft_negatives)):

    # Premise becomes the anchor
    train_dataset["anchor"].append(row["premise"])

    # Entailment hypothesis becomes the positive example
    train_dataset["positive"].append(row["hypothesis"])

    # Random hypothesis becomes the negative example
    train_dataset["negative"].append(soft_negative)

# Convert the dictionary into a Hugging Face Dataset
train_dataset = Dataset.from_dict(train_dataset)

In [ ]:
# Import the evaluator used to measure the quality of sentence embeddings
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the STS-B (Semantic Textual Similarity Benchmark) validation dataset
val_sts = load_dataset("glue", "stsb", split="validation")

# Create an evaluator that compares the model's predicted similarity
# with the actual human similarity scores
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence from each sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence from each sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize similarity scores from the range 0–5 to 0–1
    scores=[
        score / 5
        for score in val_sts["label"]
    ],

    # Use cosine similarity to compare sentence embeddings
    main_similarity="cosine"
)

In [ ]:
# Import different loss functions and the SentenceTransformer model
from sentence_transformers import losses, SentenceTransformer

# Import the trainer used to train SentenceTransformer models
from sentence_transformers.trainer import SentenceTransformerTrainer

# Import training configuration arguments
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Load the pretrained BERT model as the base embedding model
embedding_model = SentenceTransformer("bert-base-uncased")

# Define the loss function
# MultipleNegativesRankingLoss uses all other examples in the batch
# as negative examples while the matching pair is treated as the positive example
train_loss = losses.MultipleNegativesRankingLoss(
    model=embedding_model
)

# Define the training configuration
args = SentenceTransformerTrainingArguments(

    # Folder where the trained model will be saved
    output_dir="mnrloss_embedding_model",

    # Number of training epochs
    num_train_epochs=1,

    # Training batch size
    per_device_train_batch_size=32,

    # Evaluation batch size
    per_device_eval_batch_size=32,

    # Number of warmup steps for the learning rate scheduler
    warmup_steps=100,

    # Enable mixed precision (FP16) training to speed up training
    fp16=True,

    # Evaluate the model every 100 training steps
    eval_steps=100,

    # Print training logs every 100 steps
    logging_steps=100,
)

# Create the SentenceTransformer trainer
trainer = SentenceTransformerTrainer(

    # Sentence embedding model to train
    model=embedding_model,

    # Training configuration
    args=args,

    # Training dataset
    train_dataset=train_dataset,

    # Loss function used during training
    loss=train_loss,

    # Evaluator used to measure model performance
    evaluator=evaluator
)

# Start training the embedding model
trainer.train()

# Evaluate the trained model on the STS-B validation dataset
evaluator(embedding_model)

In [ ]:
# Import the function to load datasets from Hugging Face
from datasets import load_dataset

# Import the evaluator used to measure sentence embedding quality
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the first 50,000 examples from the MNLI training dataset
# MNLI labels:
# 0 = Entailment
# 1 = Neutral
# 2 = Contradiction
train_dataset = load_dataset(
    "glue",
    "mnli",
    split="train"
).select(range(50_000))

# Remove the unnecessary index column
train_dataset = train_dataset.remove_columns("idx")

# Load the STS-B (Semantic Textual Similarity Benchmark) validation dataset
# This dataset is used only for evaluating the embedding model
val_sts = load_dataset(
    "glue",
    "stsb",
    split="validation"
)

# Create an evaluator for measuring sentence similarity
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence from each sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence from each sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize similarity scores from the range 0–5 to 0–1
    scores=[
        score / 5
        for score in val_sts["label"]
    ],

    # Compare embeddings using cosine similarity
    main_similarity="cosine"
)

In [ ]:
# Import different loss functions and the SentenceTransformer model
from sentence_transformers import losses, SentenceTransformer

# Import the trainer used to train SentenceTransformer models
from sentence_transformers.trainer import SentenceTransformerTrainer

# Import the class used to define training configurations
from sentence_transformers.training_args import SentenceTransformerTrainingArguments


# Load the pretrained all-MiniLM-L6-v2 sentence embedding model
# This model is already optimized for generating high-quality sentence embeddings
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


# Define the loss function
# MultipleNegativesRankingLoss uses one positive pair and automatically
# treats all other examples in the batch as negative examples
train_loss = losses.MultipleNegativesRankingLoss(
    model=embedding_model
)


# Define all training settings
args = SentenceTransformerTrainingArguments(

    # Folder where the fine-tuned model will be saved
    output_dir="finetuned_embedding_model",

    # Number of complete passes through the training dataset
    num_train_epochs=1,

    # Number of training examples processed in one batch
    per_device_train_batch_size=32,

    # Number of evaluation examples processed in one batch
    per_device_eval_batch_size=32,

    # Number of warmup steps before reaching the full learning rate
    warmup_steps=100,

    # Enable FP16 mixed-precision training to reduce memory usage
    # and speed up training on supported GPUs
    fp16=True,

    # Evaluate the model after every 100 training steps
    eval_steps=100,

    # Print training logs after every 100 steps
    logging_steps=100,
)


# Create the trainer that manages the complete training process
trainer = SentenceTransformerTrainer(

    # Sentence embedding model to fine-tune
    model=embedding_model,

    # Training configuration
    args=args,

    # Dataset used for training
    train_dataset=train_dataset,

    # Loss function used to optimize the embeddings
    loss=train_loss,

    # Evaluator used to measure model performance during training
    evaluator=evaluator
)


# Start fine-tuning the sentence embedding model
trainer.train()

In [ ]:
# Evaluate the performance of the trained sentence embedding model
evaluator(embedding_model)

In [ ]:
# Import the Pandas library for handling tabular data
import pandas as pd

# Import tqdm to display a progress bar
from tqdm import tqdm

# Import functions to load datasets and create custom datasets
from datasets import load_dataset, Dataset

# Import InputExample to create training examples
from sentence_transformers import InputExample

# Import a DataLoader that avoids duplicate sentence pairs in each batch
from sentence_transformers.datasets import NoDuplicatesDataLoader


# Load the first 10,000 examples from the MNLI training dataset
dataset = load_dataset(
    "glue",
    "mnli",
    split="train"
).select(range(10_000))


# Convert MNLI labels into binary labels
# Original:
# 0 = Entailment
# 1 = Neutral
# 2 = Contradiction
#
# New:
# Entailment = 1
# Neutral + Contradiction = 0
mapping = {
    2: 0,
    1: 0,
    0: 1
}


# Create InputExample objects for every sentence pair
gold_examples = [

    # Store sentence1, sentence2, and their binary label
    InputExample(
        texts=[
            row["premise"],
            row["hypothesis"]
        ],
        label=mapping[row["label"]]
    )

    # Repeat for every row in the dataset
    for row in tqdm(dataset)
]


# Create a DataLoader
# NoDuplicatesDataLoader ensures duplicate sentence pairs
# are not placed in the same training batch
gold_dataloader = NoDuplicatesDataLoader(
    gold_examples,
    batch_size=32
)


# Convert the dataset into a Pandas DataFrame
# This makes it easier to inspect, filter, and manipulate the data
gold = pd.DataFrame(

    {
        # First sentence
        "sentence1": dataset["premise"],

        # Second sentence
        "sentence2": dataset["hypothesis"],

        # Binary labels
        "label": [
            mapping[label]
            for label in dataset["label"]
        ]
    }
)

In [ ]:
# Import the CrossEncoder model
# A CrossEncoder processes both sentences together and predicts a class or score
from sentence_transformers.cross_encoder import CrossEncoder


# Load the pretrained BERT model as a CrossEncoder
# num_labels=2 means this is a binary classification task
# (0 = Not Entailment, 1 = Entailment)
cross_encoder = CrossEncoder(
    "bert-base-uncased",
    num_labels=2
)


# Fine-tune the CrossEncoder on the gold dataset
cross_encoder.fit(

    # Training DataLoader containing sentence pairs and labels
    train_dataloader=gold_dataloader,

    # Train for one complete pass through the dataset
    epochs=1,

    # Display a progress bar during training
    show_progress_bar=True,

    # Number of warmup steps for the learning rate scheduler
    warmup_steps=100,

    # Disable Automatic Mixed Precision (AMP)
    # Set to True if you want faster training on supported GPUs
    use_amp=False
)

In [ ]:
# Import the function to load datasets from Hugging Face
from datasets import load_dataset

# Load the next 40,000 examples from the MNLI training dataset
# (Examples from index 10,000 to 49,999)
# These examples will become the "silver" dataset
silver = load_dataset(
    "glue",
    "mnli",
    split="train"
).select(range(10_000, 50_000))

# Create sentence pairs by combining each premise with its corresponding hypothesis
# zip() pairs the first premise with the first hypothesis,
# the second premise with the second hypothesis, and so on
# list() converts the zip object into a list of tuples
pairs = list(
    zip(
        silver["premise"],
        silver["hypothesis"]
    )
)

In [ ]:
# Import NumPy for numerical operations
import numpy as np

# Use the fine-tuned CrossEncoder to predict labels
# for every sentence pair in the silver dataset
output = cross_encoder.predict(

    # List of (premise, hypothesis) sentence pairs
    pairs,

    # Convert model outputs (logits) into probabilities
    apply_softmax=True,

    # Display a progress bar while making predictions
    show_progress_bar=True
)

# Create a new Pandas DataFrame containing
# the sentence pairs and the predicted labels
silver = pd.DataFrame(

    {
        # First sentence (premise)
        "sentence1": silver["premise"],

        # Second sentence (hypothesis)
        "sentence2": silver["hypothesis"],

        # Predicted class label
        # argmax() selects the class with the highest probability
        "label": np.argmax(output, axis=1)
    }
)

In [ ]:
# Combine the Gold dataset (human-labeled) and Silver dataset (model-labeled)
# into a single Pandas DataFrame
data = pd.concat(
    [gold, silver],          # List of DataFrames to combine
    ignore_index=True,       # Create new row indices (0, 1, 2, ...)
    axis=0                   # Stack rows vertically
)

# Remove duplicate sentence pairs .If the same sentence pair appears multiple times,
# keep only the first occurrence
data = data.drop_duplicates(
    subset=["sentence1", "sentence2"],
    keep="first"
)

# Convert the Pandas DataFrame into a Hugging Face Dataset
# preserve_index=False prevents the DataFrame index from becoming
# an extra column in the Dataset
train_dataset = Dataset.from_pandas(
    data,
    preserve_index=False
)

In [ ]:
# Import the evaluator used to evaluate sentence embedding models
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the STS-B (Semantic Textual Similarity Benchmark) validation dataset
# This dataset contains pairs of sentences and their human similarity scores
val_sts = load_dataset(
    "glue",
    "stsb",
    split="validation"
)

# Create an evaluator for measuring the quality of sentence embeddings
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence in every sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence in every sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize similarity scores from 0–5 to 0–1
    # Example:
    # 5.0 -> 1.0
    # 4.0 -> 0.8
    # 2.5 -> 0.5
    scores=[
        score / 5
        for score in val_sts["label"]
    ],

    # Use cosine similarity to compare the generated sentence embeddings
    main_similarity="cosine"
)

In [ ]:
# Import different loss functions and the SentenceTransformer model
from sentence_transformers import losses, SentenceTransformer

# Import the trainer used to fine-tune the SentenceTransformer model
from sentence_transformers.trainer import SentenceTransformerTrainer

# Import the class used to define training configurations
from sentence_transformers.training_args import SentenceTransformerTrainingArguments


# Load the pretrained BERT model
# This model will be fine-tuned to generate better sentence embeddings
embedding_model = SentenceTransformer("bert-base-uncased")


# Define the loss function
# CosineSimilarityLoss trains the model so that the cosine similarity
# between two sentence embeddings matches their similarity label
train_loss = losses.CosineSimilarityLoss(
    model=embedding_model
)


# Define all training settings
args = SentenceTransformerTrainingArguments(

    # Folder where the fine-tuned model will be saved
    output_dir="augmented_embedding_model",

    # Train for one complete pass through the dataset
    num_train_epochs=1,

    # Number of training examples processed at once
    per_device_train_batch_size=32,

    # Number of evaluation examples processed at once
    per_device_eval_batch_size=32,

    # Number of warmup steps before reaching the full learning rate
    warmup_steps=100,

    # Enable mixed precision (FP16) training for faster GPU training
    fp16=True,

    # Evaluate the model after every 100 training steps
    eval_steps=100,

    # Print training logs after every 100 steps
    logging_steps=100,
)


# Create the trainer that manages the complete training process
trainer = SentenceTransformerTrainer(

    # Sentence embedding model to fine-tune
    model=embedding_model,

    # Training configuration
    args=args,

    # Training dataset (Gold + Silver combined dataset)
    train_dataset=train_dataset,

    # Loss function used during optimization
    loss=train_loss,

    # Evaluator used to measure model performance during training
    evaluator=evaluator
)


# Start fine-tuning the sentence embedding model
trainer.train()

In [ ]:
# Evaluate the fine-tuned sentence embedding model
evaluator(embedding_model)

In [ ]:
# Import the Natural Language Toolkit (NLTK) library
# NLTK provides many tools for text preprocessing and NLP tasks
import nltk

# Download the "punkt" tokenizer package
# This tokenizer is used to split text into sentences and words
nltk.download("punkt")

In [ ]:
# Import tqdm to display a progress bar
from tqdm import tqdm

# Import Hugging Face Dataset utilities
from datasets import Dataset, load_dataset

# Import the dataset class that automatically creates noisy sentence pairs
from sentence_transformers.datasets import DenoisingAutoEncoderDataset

# Import the NLTK library
import nltk

# Download the tokenizer resource required for sentence tokenization
nltk.download("punkt_tab")


# Load the first 25,000 examples from the MNLI training dataset
mnli = load_dataset(
    "glue",
    "mnli",
    split="train"
).select(range(25_000))


# Combine all premise sentences and hypothesis sentences
# into one large list of sentences
flat_sentences = mnli["premise"] + mnli["hypothesis"]


# Create a denoising dataset
# list(set(...)) removes duplicate sentences
# DenoisingAutoEncoderDataset automatically creates:Noisy sentence  -> Original sentence
damaged_data = DenoisingAutoEncoderDataset(
    list(set(flat_sentences))
)


# Create an empty dictionary that will store
# noisy sentences and their original versions
train_dataset = {
    "damaged_sentence": [],
    "original_sentence": []
}


# Go through every noisy sentence pair
for data in tqdm(damaged_data):

    # Store the damaged (corrupted) sentence
    train_dataset["damaged_sentence"].append(
        data.texts[0]
    )

    # Store the original sentence
    train_dataset["original_sentence"].append(
        data.texts[1]
    )


# Convert the dictionary into a Hugging Face Dataset
train_dataset = Dataset.from_dict(train_dataset)

In [ ]:
# It returns the first example (row 0) from the Hugging Face Dataset
# Damahed to original sentence
#Ex : The output will look like
#  {
#     'damaged_sentence': 'cat sleeping sofa',
#     'original_sentence': 'The cat is sleeping on the sofa.'
# }
train_dataset[0]

In [ ]:
# Import the evaluator used to measure the quality of sentence embeddings
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load the STS-B (Semantic Textual Similarity Benchmark) validation dataset
# This dataset contains sentence pairs along with human similarity scores
val_sts = load_dataset(
    "glue",
    "stsb",
    split="validation"
)

# Create an evaluator to test the sentence embedding model
evaluator = EmbeddingSimilarityEvaluator(

    # First sentence from each sentence pair
    sentences1=val_sts["sentence1"],

    # Second sentence from each sentence pair
    sentences2=val_sts["sentence2"],

    # Normalize similarity scores from the range 0–5 to 0–1
    # Example:
    # 5.0 → 1.0
    # 4.0 → 0.8
    # 2.5 → 0.5
    scores=[
        score / 5
        for score in val_sts["label"]
    ],

    # Compare sentence embeddings using cosine similarity
    main_similarity="cosine"
)

In [ ]:
# Import the building blocks used to manually create a SentenceTransformer model
from sentence_transformers import models, SentenceTransformer


# Load the pretrained BERT Transformer model
# This model converts input text into contextual word embeddings
word_embedding_model = models.Transformer(
    "bert-base-uncased"
)


# Create a pooling layer
# get_word_embedding_dimension() returns the size of each word embedding
# "cls" means use the embedding of the special [CLS] token
# as the representation of the entire sentence
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    "cls"
)


# Combine the Transformer model and the Pooling layer
# to create a complete SentenceTransformer model
embedding_model = SentenceTransformer(
    modules=[
        word_embedding_model,
        pooling_model
    ]
)

In [ ]:
# Import the available loss functions from Sentence Transformers
from sentence_transformers import losses

# Create a Denoising AutoEncoder (DAE) loss
# The model learns to reconstruct the original sentence
# from a damaged (noisy) version of the sentence
train_loss = losses.DenoisingAutoEncoderLoss(

    # Sentence embedding model (encoder)
    embedding_model,

    # Share (tie) the encoder and decoder weights
    # This reduces the number of parameters and often improves training
    tie_encoder_decoder=True
)

# Move the decoder part of the Denoising AutoEncoder
# to the GPU for faster training
train_loss.decoder = train_loss.decoder.to("cuda")

In [ ]:
# Import the trainer used to fine-tune SentenceTransformer models
from sentence_transformers.trainer import SentenceTransformerTrainer

# Import the class used to configure the training process
from sentence_transformers.training_args import SentenceTransformerTrainingArguments


# Define all the training settings
args = SentenceTransformerTrainingArguments(

    # Folder where the trained TSDAE model will be saved
    # TSDAE (Transformer-based Sequential Denoising AutoEncoder) is a self-supervised sentence embedding method that uses a Transformer encoder and a decoder to learn sentence representations by reconstructing the original sentence from a corrupted (noisy) version. It does not require manually labeled training data.
    output_dir="tsdae_embedding_model",

    # Number of complete passes over the training dataset
    num_train_epochs=1,

    # Number of training examples processed in one batch
    per_device_train_batch_size=16,

    # Number of validation examples processed in one batch
    per_device_eval_batch_size=16,

    # Number of warmup steps before reaching the full learning rate
    warmup_steps=100,

    # Enable mixed precision (FP16) training for faster GPU training
    fp16=True,

    # Evaluate the model after every 100 training steps
    eval_steps=100,

    # Print training logs after every 100 training steps
    logging_steps=100,
)


# Create the trainer that manages the complete training process
trainer = SentenceTransformerTrainer(

    # Sentence embedding model (Encoder)
    model=embedding_model,

    # Training configuration
    args=args,

    # Dataset containing damaged and original sentences
    train_dataset=train_dataset,

    # Denoising AutoEncoder loss function
    loss=train_loss,

    # Evaluator used to measure embedding quality during training
    evaluator=evaluator
)


# Start training the TSDAE sentence embedding model
trainer.train()

In [ ]:
# Evaluate our trained model
evaluator(embedding_model)

In [ ]:
# Import the function used to download datasets from Hugging Face
from datasets import load_dataset

# Load the CoNLL-2003 dataset
# This is a benchmark dataset for Named Entity Recognition (NER)
# It contains sentences where every word is labeled with an entity tag
# such as Person (PER), Location (LOC), Organization (ORG), or Miscellaneous (MISC)
# trust_remote_code=True allows Hugging Face to execute any custom dataset loading code required by this dataset.
dataset = load_dataset(
    "conll2003",
    trust_remote_code=True
)

In [ ]:
# Select the 849th example (index 848) from the training dataset
# dataset["train"] accesses the training split
# [848] retrieves a single example from that split
example = dataset["train"][848]

# Display the selected example
example

In [ ]:
# Dictionary that maps each NER (Named Entity Recognition) label to a unique integer ID
# Models cannot directly understand text labels, so we convert labels into numbers

label2id = {
    "O": 0,        # O = Outside any named entity (normal words that are not entities)
    "B-PER": 1,    # B-PER = Beginning of a Person name
    "I-PER": 2,    # I-PER = Inside a Person name

    "B-ORG": 3,    # B-ORG = Beginning of an Organization name
    "I-ORG": 4,    # I-ORG = Inside an Organization name

    "B-LOC": 5,    # B-LOC = Beginning of a Location name
    "I-LOC": 6,    # I-LOC = Inside a Location name

    "B-MISC": 7,   # B-MISC = Beginning of Miscellaneous entity
    "I-MISC": 8    # I-MISC = Inside a Miscellaneous entity
}


# Reverse mapping: Converts numerical IDs back into their original label names
# Useful when the model predicts numbers and we need human-readable labels

id2label = {
    index: label       # Create key-value pairs where ID becomes key and label becomes value
    for label, index in label2id.items()
}

In [ ]:
# Import classes required for loading a tokenizer and a token classification model
from transformers import AutoModelForTokenClassification, AutoTokenizer


# Load the tokenizer for the pre-trained BERT model
# Tokenizer converts input text into tokens and token IDs that BERT can understand
# "bert-base-cased" keeps the original capitalization of words
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


# Load BERT model with a token classification head for NER task
# Token classification means the model predicts one label for each token/word
model = AutoModelForTokenClassification.from_pretrained(

    "bert-base-cased",


    # Number of output classes the model needs to predict
    num_labels=len(id2label),


    # Mapping from label ID to label name
    # Used to convert model output numbers back into readable labels
    # Example: 1 → "B-PER"
    id2label=id2label,


    # Mapping from label name to label ID
    # Used during training to convert labels into numbers
    # Example: "B-PER" → 1
    label2id=label2id
)

In [ ]:
# Tokenize the individual words (tokens) from the input example
# is_split_into_words=True tells the tokenizer that the input is already split into words
# The tokenizer may further split words into smaller sub-tokens because BERT uses WordPiece tokenization
token_ids = tokenizer(
    example["tokens"],
    is_split_into_words=True
)["input_ids"]


# Convert numerical token IDs back into readable sub-token strings
# This helps us see how BERT actually represents the words internally
sub_tokens = tokenizer.convert_ids_to_tokens(token_ids)


# Display the generated sub-tokens
sub_tokens

In [ ]:
def align_labels(examples):

    # Tokenize the input words
    # is_split_into_words=True means the input is already separated into words
    # Example: ["John", "Smith"] instead of "John Smith"
    # truncation=True prevents sequences longer than BERT's maximum length
    token_ids = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )


    # Get the original NER labels for each word
    labels = examples["ner_tags"]


    # Store the final aligned labels for all examples
    all_updated_labels = []


    # Process each sentence/example in the batch
    for index, current_example_labels in enumerate(labels):

        # Get the mapping between sub-tokens and original words
        # It tells which word each token came from
        #
        # Example:
        # Words:
        # ["John", "playing"]
        # Tokens:
        # [CLS] John play ##ing [SEP]
        # word_ids:
        # [None, 0, 1, 1, None]
        #
        word_ids = token_ids.word_ids(batch_index=index)


        # Keeps track of the previous word index
        # Used to detect whether the current token belongs to a new word or the same word as the previous token
        previous_word_idx = None


        # Store labels after aligning them with sub-tokens
        label_ids = []


        # Go through every token and assign a label
        for word_idx in word_ids:
            # These tokens do not have NER labels,
            # so we assign -100.
            # PyTorch ignores -100 while calculating loss.
            if word_idx is None:
                label_ids.append(-100)


            # If this is the first sub-token of a word
            # Use the original word's NER label
            elif word_idx != previous_word_idx:
                label_ids.append(current_example_labels[word_idx])


            else:
                # This means the token belongs to the same word
                # Get the original label of this word
                current_label_for_word = current_example_labels[word_idx]


                # Check whether the label is B-XXX
                #
                # Label IDs:
                # B-PER  = 1 (odd)
                # B-ORG  = 3 (odd)
                # B-LOC  = 5 (odd)
                # I labels are even numbers.
                if current_label_for_word % 2 == 1:
                    # Convert B-XXX label to I-XXX label
                    # Example:
                    # B-PER (1) becomes I-PER (2)
                    # Because continuation sub-tokens cannot be the beginning
                    # of an entity.
                    label_ids.append(current_label_for_word + 1)


                else:

                    # For O labels or already I-XXX labels,
                    # ignore additional sub-tokens by assigning -100 Loss will not be calculated for these tokens.
                    label_ids.append(-100)


            # Update previous word index
            previous_word_idx = word_idx


        # Add the aligned labels of this example to the final list
        all_updated_labels.append(label_ids)


    # Add the new aligned labels to the tokenized output
    # The model will use these labels during training
    token_ids["labels"] = all_updated_labels


    # Return tokenized inputs along with aligned labels
    return token_ids



# Apply the align_labels function to the complete dataset
# batched=True means multiple examples are processed together for efficiency
tokenized = dataset.map(
    align_labels,
    batched=True
)